In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, StandardScaler, MinMaxScaler
from sklearn.compose import  ColumnTransformer
from sklearn.metrics import r2_score, log_loss, accuracy_score, roc_auc_score

from sklearn.discriminant_analysis import LinearDiscriminantAnalysis, QuadraticDiscriminantAnalysis

from tqdm import tqdm  # Provides the progress of model running

import os
os.chdir("C:/Users/PGCP-AI/Downloads/playground-series-s6e3") #fetch the dataset from the folder

### Sonar Dataset

In [2]:
sonar = pd.read_csv('D:/Machine_Learning/Cases/Sonar/Sonar.csv')
sonar.head()

,V1,V2,V3,V4,V5,V6,V7,V8,V9,V10,...,V52,V53,V54,V55,V56,V57,V58,V59,V60,Class
0,0.0200,0.0371,0.0428,0.0207,0.0954,0.0986,0.1539,0.1601,0.3109,0.2111,...,0.0027,0.0065,0.0159,0.0072,0.0167,0.0180,0.0084,0.0090,0.0032,R
1,0.0453,0.0523,0.0843,0.0689,0.1183,0.2583,0.2156,0.3481,0.3337,0.2872,...,0.0084,0.0089,0.0048,0.0094,0.0191,0.0140,0.0049,0.0052,0.0044,R
2,0.0262,0.0582,0.1099,0.1083,0.0974,0.2280,0.2431,0.3771,0.5598,0.6194,...,0.0232,0.0166,0.0095,0.0180,0.0244,0.0316,0.0164,0.0095,0.0078,R
3,0.0100,0.0171,0.0623,0.0205,0.0205,0.0368,0.1098,0.1276,0.0598,0.1264,...,0.0121,0.0036,0.0150,0.0085,0.0073,0.0050,0.0044,0.0040,0.0117,R
4,0.0762,0.0666,0.0481,0.0394,0.0590,0.0649,0.1209,0.2467,0.3564,0.4459,...,0.0031,0.0054,0.0105,0.0110,0.0015,0.0072,0.0048,0.0107,0.0094,R


In [3]:
le = LabelEncoder()
sonar['Class'] = le.fit_transform(sonar['Class'])
X , y = sonar.drop('Class', axis=1), sonar['Class']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size= 0.3, random_state=26, stratify= sonar['Class'])


In [4]:
lda = LinearDiscriminantAnalysis()
lda.fit(X_train,y_train) 

y_pred_prob = lda.predict_proba(X_test) 
print("ROC AUC =", roc_auc_score(y_test, y_pred_prob[:,1]))
    
y_pred = lda.predict(X_test)
print("Accuracy Score=", accuracy_score(y_test, y_pred))

ROC AUC = 0.7941176470588235
Accuracy Score= 0.7301587301587301


In [5]:
qda = QuadraticDiscriminantAnalysis(reg_param=1)  # reg_param(regularization parameter) = used to change the matrices from no_invertible to invertible
                                                      # (rank is low so inverse not able to calculate so covariance added to increase rank)
qda.fit(X_train,y_train) 

y_pred_prob = qda.predict_proba(X_test) 
print("ROC AUC =", roc_auc_score(y_test, y_pred_prob[:,1]))
    
y_pred = qda.predict(X_test)
print("Accuracy Score=", accuracy_score(y_test, y_pred))

ROC AUC = 0.8062880324543611
Accuracy Score= 0.6984126984126984


In [6]:
# Extra work not needed in exam
regparam=np.linspace(0.001,1,100)
scores=[]
for p in regparam:
    qda = QuadraticDiscriminantAnalysis(reg_param=p)
    qda.fit(X_train, y_train)
    
    y_pred_prob = qda.predict_proba(X_test)
    y_pred = qda.predict(X_test)
    
    scores.append([p, roc_auc_score(y_test, y_pred_prob[:,1]), accuracy_score(y_test, y_pred)])
    
df_scores = pd.DataFrame(scores, columns=['regparam', 'ROCAUC', 'Accuracy Score' ])
df_scores.sort_values(['Accuracy Score','ROCAUC'], ascending=False).head()

,regparam,ROCAUC,Accuracy Score
2,0.021182,0.905680,0.873016
10,0.101909,0.915822,0.857143
7,0.071636,0.915822,0.857143
9,0.091818,0.914807,0.857143
3,0.031273,0.913793,0.857143


### Image Segmentation Dataset

In [7]:
os.chdir("D:/Machine_Learning/Cases/Image_Segmentation") #fetch the dataset from the folder
img = pd.read_csv("Image_Segmentation.csv")
img


,Class,region.centroid.col,region.centroid.row,region.pixel.count,short.line.density.5,short.line.density.2,vedge.mean,vegde.sd,hedge.mean,hedge.sd,intensity.mean,rawred.mean,rawblue.mean,rawgreen.mean,exred.mean,exblue.mean,exgreen.mean,value.mean,saturation.mean,hue-mean
0,BRICKFACE,188,133,9,0.000000,0.0,0.333333,0.266667,0.500000,0.077778,6.666666,8.333334,7.777778,3.888889,5.000000,3.333333,-8.333333,8.444445,0.538580,-0.924817
1,BRICKFACE,105,139,9,0.000000,0.0,0.277778,0.107407,0.833333,0.522222,6.111111,7.555555,7.222222,3.555556,4.333334,3.333333,-7.666666,7.555555,0.532628,-0.965946
2,BRICKFACE,34,137,9,0.000000,0.0,0.500000,0.166667,1.111111,0.474074,5.851852,7.777778,6.444445,3.333333,5.777778,1.777778,-7.555555,7.777778,0.573633,-0.744272
3,BRICKFACE,39,111,9,0.000000,0.0,0.722222,0.374074,0.888889,0.429629,6.037037,7.000000,7.666666,3.444444,2.888889,4.888889,-7.777778,7.888889,0.562919,-1.175773
4,BRICKFACE,16,128,9,0.000000,0.0,0.500000,0.077778,0.666667,0.311111,5.555555,6.888889,6.666666,3.111111,4.000000,3.333333,-7.333334,7.111111,0.561508,-0.985811
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
204,GRASS,36,243,9,0.111111,0.0,1.888889,1.851851,2.000000,0.711110,13.333333,9.888889,12.111111,18.000000,-10.333333,-3.666667,14.000000,18.000000,0.452229,2.368311
205,GRASS,186,218,9,0.000000,0.0,1.166667,0.744444,1.166667,0.655555,13.703704,10.666667,12.666667,17.777779,-9.111111,-3.111111,12.222222,17.777779,0.401347,2.382684
206,GRASS,197,236,9,0.000000,0.0,2.444444,6.829628,3.333333,7.599998,16.074074,13.111111,16.666668,18.444445,-8.888889,1.777778,7.111111,18.555555,0.292729,2.789800
207,GRASS,208,240,9,0.111111,0.0,1.055556,0.862963,2.444444,5.007407,14.148149,10.888889,13.000000,18.555555,-9.777778,-3.444444,13.222222,18.555555,0.421621,2.392487


In [8]:
le = LabelEncoder()
img['Class'] = le.fit_transform(img['Class'])
X , y = img.drop('Class', axis=1), img['Class']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size= 0.3, random_state=26, stratify= img['Class'])

In [9]:
lda = LinearDiscriminantAnalysis()
lda.fit(X_train,y_train) 

y_pred_prob = lda.predict_proba(X_test) 
print("Log_Loss =", log_loss(y_test, y_pred_prob))
    
y_pred = lda.predict(X_test)
print("Accuracy Score=", accuracy_score(y_test, y_pred))

Log_Loss = 1.3575030968052337
Accuracy Score= 0.9206349206349206


In [10]:
qda = QuadraticDiscriminantAnalysis(reg_param=1) 
qda.fit(X_train,y_train) 

y_pred_prob = qda.predict_proba(X_test) 
print("Log_Loss =", log_loss(y_test, y_pred_prob))
    
y_pred = qda.predict(X_test)
print("Accuracy Score=", accuracy_score(y_test, y_pred))

Log_Loss = 9.726065253372449
Accuracy Score= 0.7301587301587301


In [11]:
# Extra work not needed in exam
regparam=np.linspace(0.001,1,100)
scores=[]
for p in regparam:
    qda = QuadraticDiscriminantAnalysis(reg_param=p)
    qda.fit(X_train, y_train)
    
    y_pred_prob = qda.predict_proba(X_test)
    y_pred = qda.predict(X_test)
    
    scores.append([p, log_loss(y_test, y_pred_prob), accuracy_score(y_test, y_pred)])
    
df_scores = pd.DataFrame(scores, columns=['regparam', 'Log_Loss', 'Accuracy Score' ])
df_scores.sort_values(['Log_Loss','Accuracy Score'], ascending=[True,False]).head()

,regparam,Log_Loss,Accuracy Score
1,0.011091,1.860214,0.888889
80,0.808273,1.987356,0.920635
81,0.818364,1.987469,0.920635
79,0.798182,1.987637,0.920635
82,0.828455,1.988038,0.920635
